# P04 · Notebook 33 · The four simple families on DCASE 2025 Task 2

**Why.** The deck presents five families of detectors. Families 1 to 4 were only tried on four pumps; family 5 alone went to the public
tests. This notebook runs families 1 to 4 on DCASE 2025, with the settings of the pump runs unchanged, so that all five can be set
side by side on the same test. The rules were written first: `results/mimii_beats/RULES_simple_families_written_before_running.md`.

**What it does.** For every machine type (each has one section): fit each family on that unit's normal training clips, score its test clips, and compute
the official score (harmonic mean) exactly as the official evaluator does. The evaluation labels are read once, at the end, after every score is saved.

| | Family | What it reads | Detector |
|---|---|---|---|
| 1 | A limit on one number | the clip's loudness | distance from the usual loudness |
| 2 | Distance from the normal cloud | 128 numbers per clip | PCA, T² + SPE |
| 3 | Isolation | 128 numbers per clip | Isolation Forest |
| 4 | Autoencoder | the log-mel picture, 5 slices at a time | reconstruction error |

**How to run.** Runtime › Run all. **No GPU is needed** (a CPU runtime is enough). About 7 gb of downloads from zenodo (the 23 zips of notebook 30), and the evaluator's repository; a zip is deleted once its
clips are stored. Expected time: **about 1 hour** (an estimate: one network took 2.2 minutes on a 2-CPU test machine; Colab may be slower). Almost all of it is family 4, which trains one small network per unit
(15 units). Every finished unit is saved in Drive › `P04_SimpleFamilies_2025`, so if the session stops, **Run all again continues** where it stopped
(the downloads are repeated, the trained units are not).

**Send back** the zip that downloads at the end (`p04_n33_results.zip`).

In [ ]:
# 1 · Setup (no GPU is needed: a CPU runtime is enough)
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, csv, warnings
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, WORK = Path("/content/drive/MyDrive") / "P04_SimpleFamilies_2025", Path("/content/work")
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_SimpleFamilies_2025")), Path(os.environ.get("P04_WORK", "work"))
for p in (OUT, WORK, OUT / "scores"):
    p.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print("free local disk", round(shutil.disk_usage(WORK).free / 1e9), "GB · free Drive", round(shutil.disk_usage(OUT).free / 1e9, 1), "GB · CPUs", os.cpu_count())


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · The files, and what was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_simple_families_written_before_running.md"
# DCASE 2025 Task 2: the file lists, MD5s and evaluator of notebook 30
ZD, ZX, ZE = "https://zenodo.org/records/15097779/files/", "https://zenodo.org/records/15392814/files/", "https://zenodo.org/records/15519362/files/"
TYPES_DEV = ["ToyCar", "ToyTrain", "bearing", "fan", "gearbox", "slider", "valve"]
TYPES_EVAL = ["AutoTrash", "BandSealer", "CoffeeGrinder", "HomeCamera", "Polisher", "ScrewFeeder", "ToyPet", "ToyRCCar"]
DMD5 = {"ToyCar": "16bd2069065281ea7ac0e96894f8592c", "ToyTrain": "c2524ff503bc1631ca308dc840bc5010", "bearing": "b3a13693b69b477ce814db8b40783955",
        "fan": "61b6aae6d5ebb8cb937faa86741fbf47", "gearbox": "9542948d980fd575426f60f2a41060b8", "slider": "6c1f2d2ef0bc1a960db5938adb8425d1",
        "valve": "5640c0dc03982107ef354fe9480c9a83"}
XMD5 = {"AutoTrash": "0d89113bee9837402c609cb466c4a953", "BandSealer": "0e43610e140e34e0187b25f09d1f447b", "CoffeeGrinder": "9f39058361fde38fd5e24a52262d5977",
        "HomeCamera": "f81a89d772b1454af476059cb9800788", "Polisher": "09e62aea7f2a82b417e7c388c1060e92", "ScrewFeeder": "4a30f5aa52f019cac1d8653931ac4f44",
        "ToyPet": "0684d7185b4feea26b8479324f9c6573", "ToyRCCar": "74dc1d920983255d53b5214b0429b4d8"}
EMD5 = {"AutoTrash": "b33216913b71328d813bcbb3ec7415d7", "BandSealer": "78a582ca8092b35861542b1c34268952", "CoffeeGrinder": "359951de42956ef9a853dec6121a9517",
        "HomeCamera": "1b2e09f3283be0531fafeb5a66b58dd4", "Polisher": "7e50e4296cd7bd0ab94df37673df313e", "ScrewFeeder": "88035983d52753cf0bf2c9a692740222",
        "ToyPet": "c38c60b36b3cfdff5429fb6673eb9c57", "ToyRCCar": "1a6797436fe20d1a86224e65c8c19563"}
DC_DEV = {t_: (ZD + f"dev_{t_}.zip?download=1", f"dev_{t_}.zip", h) for t_, h in DMD5.items()}
XZIPS = {t_: (ZX + f"eval_data_{t_}_train.zip?download=1", f"eval_data_{t_}_train.zip", h) for t_, h in XMD5.items()}
DC_EVAL = {t_: (ZE + f"eval_data_{t_}_test.zip?download=1", f"eval_data_{t_}_test.zip", h) for t_, h in EMD5.items()}
EV_COMMIT = "f903124a40a4cdb3faf2b2d6fa69c6a9f1794192"
EV_ARCHIVE = (f"https://github.com/nttcslab/dcase2025_task2_evaluator/archive/{EV_COMMIT}.zip", "dcase2025_task2_evaluator.zip")
EV_SCRIPT, EV_SHA = "dcase2025_task2_evaluator.py", "b20577511228d4243264fb339c06f86d2766ec50334609a3a45c335cc3b11102"
EV_TOOLS = ["tools/plot_anm_score.py", "tools/test_plots.py"]
GT_PARTS = ("ground_truth_data", "ground_truth_domain", "ground_truth_attributes")
GT_SHA = "e54a474722f1fb5d878bed257c75444251eeefae4e4dd40a0f5363f8cec68850"
SECTION = "section_00"
SR, LDN_FLOOR = 16000, 1e-6
NS_RULE = 10 * SR
NS_MIN = 1 * SR
RECORDED = {"Family 5 · the kept system (expert + trainee)": 0.5314, "Family 5 · the expert alone": 0.5922, "Family 5 · the expert with attribute classes": 0.6228,
            "first team": 0.6163, "official baseline · autoencoder, Mahalanobis": 0.5651, "official baseline · autoencoder, MSE": 0.5443}
RECORDED_NOTE = "the official score, evaluation set (eight machine types)"
if TEST:
    _dc = json.loads(os.environ["P04_TEST_DCASE"]); _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    TYPES_DEV, TYPES_EVAL = list(_dc["dev"]), list(_dc["add"])
    DC_DEV, XZIPS, DC_EVAL = ({t_: tuple(v) for t_, v in _dc[k_].items()} for k_ in ("dev", "add", "eval"))
    EV_ARCHIVE, EV_SHA, GT_SHA = tuple(_dc["evaluator_archive"]), _dc["evaluator_sha256"], _dc["ground_truth_sha256"]
    NS_RULE = _c.get("clip_samples", NS_RULE)
TYPES = TYPES_DEV + TYPES_EVAL
DEV_IDS = {t_: ([SECTION] if t_ in TYPES_DEV else []) for t_ in TYPES}
EVAL_IDS = {t_: ([SECTION] if t_ in TYPES_EVAL else []) for t_ in TYPES}

# the four simple families, with the settings of the pump runs (results/mimii/RULES_written_before_running.md), unchanged
FAMS = ["F1", "F2", "F3", "F4"]
FAM_NAME = {"F1": "Family 1 · a limit on one number (loudness)", "F2": "Family 2 · distance from the normal cloud (128 clip statistics)",
            "F3": "Family 3 · isolation (Isolation Forest, 128 clip statistics)", "F4": "Family 4 · autoencoder (reconstruction error)",
            "F1_raw": "reported beside · loudness as it is (louder = more anomalous)"}
SCORED = FAMS + ["F1_raw"]
N_FFT, HOP, N_MELS = 1024, 512, 64

In [ ]:
# 3 · The data cell of the notebook that ran on these files (its readers and its loader, unchanged)
# 9 · DCASE 2025's own files of the fifteen types (MD5 against the Zenodo records), and every clip's first 10 s in one int16 store per
# type on the local disk; gates G0(a) and G0(b). The readers are notebook 23's (through notebook 25). The loader is notebook 28's: a
# development type has one zip, an evaluation type two; the supplemental clips are not read.
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


WAV_FORMATS = Counter()                                               # the sample formats of the WAV files read in this session


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE; PCM of 16, 24 or 32 bits, or 32-bit float) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert (tag, bits) in ((1, 16), (1, 24), (1, 32), (3, 32)), f"expected PCM of 16, 24 or 32 bits or 32-bit float, got format {tag}, {bits}-bit"
    WAV_FORMATS[f"{'float' if tag == 3 else 'PCM'} {bits}-bit"] += 1
    w = bits // 8; nf = len(raw) // (w * ch); raw = raw[:nf * w * ch]
    if (tag, bits) == (1, 16):
        return np.frombuffer(raw, dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr
    if (tag, bits) == (1, 24):                                        # three bytes per sample, little-endian, signed: a 24-bit value is exact in float32
        b = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3).astype(np.int32)
        v = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16); v = np.where(v >= 1 << 23, v - (1 << 24), v)
        return (v.astype(np.float32) / 8388608.0).reshape(-1, ch), sr
    if (tag, bits) == (1, 32):
        return (np.frombuffer(raw, dtype="<i4").astype(np.float64) / 2147483648.0).astype(np.float32).reshape(-1, ch), sr
    return np.frombuffer(raw, dtype="<f4").reshape(-1, ch).astype(np.float32), sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # exact for a 16-bit WAV file; a clip of more bits is rounded to 16 (to the nearest, ties to even) and clipped to the int16 range
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores30"; STORE.mkdir(exist_ok=True)
DST, DMETA = {}, {}


def dcase_load(t):
    # every DCASE 2025 clip of type t in one int16 store of NS_RULE samples (10 s): the first 10 s of a longer clip, as the view's reference
    # code cuts it, or a shorter clip repeated from its start; a development type's train and test clips, or an evaluation type's training clips and its test clips. The supplemental clips are not read
    meta_f, arr_f = STORE / f"dc_{t}.json", STORE / f"dc_{t}.npy"
    if meta_f.exists() and arr_f.exists():
        DMETA[t] = json.loads(meta_f.read_text()); DST[t] = np.load(arr_f, mmap_mode="r")
        return
    zfs = {k_: zipfile.ZipFile(get_file(z_[t])) for k_, z_ in (("dev", DC_DEV), ("evaltrain", XZIPS), ("evaltest", DC_EVAL)) if t in z_}
    rows, skipped = [], []
    for kind, zf in zfs.items():
        for nm in sorted(x for x in zf.namelist() if x.lower().endswith(".wav")):
            parts = nm.replace("\\", "/").split("/"); base, folder = parts[-1], parts[-2]
            if "__MACOSX" in parts or base.startswith("._"):          # an archive's metadata entry, not a clip (none is expected)
                skipped.append(nm); continue
            if folder not in ("train", "test"):                       # the supplemental clips (clean machine sound or noise only) are not used
                continue
            part = folder
            # the name of a training clip and of a development test clip carries its domain and its kind; an evaluation test clip's carries neither
            said = re.match(r"section_\d+_(source|target)_(train|test)_(normal|anomaly)_", base)
            label = (1 if (said and said.group(3) == "anomaly") else 0) if (kind == "dev" and part == "test") else (0 if part == "train" else -1)
            dom = said.group(1) if said else ""
            rows.append((kind, nm, "dev" if kind == "dev" else "eval", part, re.search(r"(section_\d+)", base).group(1), label, base, dom))
    ids = {(s_, p_): sorted(set(r[4] for r in rows if r[2] == s_ and r[3] == p_)) for s_ in ("dev", "eval") for p_ in ("train", "test")}
    if not (ids[("dev", "train")] == ids[("dev", "test")] == DEV_IDS[t] and ids[("eval", "train")] == ids[("eval", "test")] == EVAL_IDS[t]):
        raise SystemExit(f"stop: G0(b) failed. The sections of {t} are not the rules': {ids}")
    assert not any(r[2] == "eval" and r[3] == "train" and "_train_normal_" not in r[6] for r in rows), "the additional training clips are normal"
    ns, lengths, _f0 = NS_RULE, Counter(), Counter(WAV_FORMATS)
    tmp = arr_f.with_name(arr_f.stem + ".part.npy")
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(rows), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zfs[rows[i][0]].read(rows[i][1])), range(len(rows)))):
        if not (sr == SR and x.shape[1] == 1 and x.shape[0] >= NS_MIN):
            raise SystemExit(f"stop: G0(b) failed. {rows[i][1]} is not a 16 kHz, one-channel clip of at least {NS_MIN / SR:g} s "
                             f"(rate {sr}, channels {x.shape[1]}, samples {x.shape[0]})")
        lengths[(f"{rows[i][2]}|{rows[i][3]}", int(x.shape[0]))] += 1
        w_ = to_int16(x[:ns, 0])
        mm[i] = w_ if len(w_) == ns else np.tile(w_, -(-ns // len(w_)))[:ns]   # a shorter clip is repeated from its start until it fills the store
    mm.flush(); del mm
    os.replace(tmp, arr_f)
    for zf in zfs.values():
        zf.close()
    DMETA[t] = {"path": [r[1] for r in rows], "name": [r[6] for r in rows], "set": [r[2] for r in rows], "part": [r[3] for r in rows],
                "mid": [r[4] for r in rows], "label": [r[5] for r in rows], "domain": [r[7] for r in rows], "samples": int(ns), "skipped_archive_entries": skipped,
                "formats": {k: int(v - _f0[k]) for k, v in sorted(WAV_FORMATS.items()) if v - _f0[k]},
                "lengths": {k: {str(n_): int(c) for (k2, n_), c in sorted(lengths.items()) if k2 == k} for k in sorted(set(k_ for k_, _ in lengths))},
                "zips": {k_: list(z_[t][1:]) for k_, z_ in (("dev", DC_DEV), ("additional", XZIPS), ("evaluation", DC_EVAL)) if t in z_}}
    meta_f.write_text(json.dumps(DMETA[t]))
    DST[t] = np.load(arr_f, mmap_mode="r")
    if IN_COLAB:
        for z in [z_[t] for z_ in (DC_DEV, XZIPS, DC_EVAL) if t in z_]:
            (WORK / z[1]).unlink(missing_ok=True)


def dc_rows(t, set_, part, q=None):
    m = DMETA[t]
    return np.array([i for i in range(len(m["name"])) if m["set"][i] == set_ and m["part"][i] == part and (q is None or m["mid"][i] == q)], dtype=np.int64)


t_data = time.time()
DCASE = {"counts": {}, "lengths": {}, "domains": {}}
for t in TYPES:
    dcase_load(t)
    _m = DMETA[t]; _mid, _dom = np.array(_m["mid"]), np.array(_m["domain"])
    DCASE["counts"][t] = {f"{s_}|{p_}": {q: int(len(dc_rows(t, s_, p_, q))) for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))}
                          for s_ in ("dev", "eval") for p_ in ("train", "test")}
    DCASE["domains"][t] = {f"{s_}|{p_}": {q: {d_: int((_dom[dc_rows(t, s_, p_, q)] == d_).sum()) for d_ in ("source", "target")}
                                          for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))} for s_, p_ in (("dev", "train"), ("dev", "test"), ("eval", "train"))}
    DCASE["lengths"][t] = _m["lengths"]; DCASE.setdefault("formats", {})[t] = _m.get("formats", "not recorded (a store of an earlier session)")
    print(f"DCASE 2025 · {t}: {len(_m['name'])} clips · training clips per section (source, target): "
          f"{({q: (v['source'], v['target']) for k_ in ('dev|train', 'eval|train') for q, v in DCASE['domains'][t][k_].items()})} · test clips per section: "
          f"{({**DCASE['counts'][t]['dev|test'], **DCASE['counts'][t]['eval|test']})} · samples per clip before the cut (set|part: samples: clips): {DCASE['lengths'][t]}", flush=True)
NS = int(DMETA[TYPES[0]]["samples"])
assert all(int(DMETA[t]["samples"]) == NS for t in TYPES) and NS == NS_RULE, "stop: a store on the local disk was not cut to the rules' length"

# G0(a) · every zip is the published file: get_file stops on another MD5, and a store keeps the names and MD5s of the zips it was built from
_want = {t: {k_: list(z_[t][1:]) for k_, z_ in (("dev", DC_DEV), ("additional", XZIPS), ("evaluation", DC_EVAL)) if t in z_} for t in TYPES}
G0 = {"a": {"ok": bool(all(DMETA[t]["zips"] == _want[t] for t in TYPES)), "zips": _want}}
print(f"G0(a) · the {sum(len(v) for v in _want.values())} zips are the published files (MD5):", G0["a"]["ok"])
if not G0["a"]["ok"]:
    raise SystemExit(f"stop: G0(a) failed. A store in {STORE} was built from other files; delete that folder and run again")


def f32(a):
    return np.asarray(a).astype(np.float32) / 32768.0


print(f"data ready · {sum(len(DMETA[t]['name']) for t in TYPES)} clips · {NS / SR:g} s each · {(time.time() - t_data) / 60:.1f} min")

In [ ]:
# 4 · The log-mel picture of every clip, with the code of the pump runs (code/p04_mimii_features.py, unchanged): 64 bands,
# slices of 1,024 readings, a new slice every 512. One float16 file per machine type on the local disk; a finished type is not redone
EPS = sys.float_info.epsilon


def hz_to_mel(f):
    """Slaney mel scale (librosa htk=False)."""
    f = np.asanyarray(f, dtype=float)
    f_sp = 200.0 / 3; mels = f / f_sp
    min_log_hz = 1000.0; min_log_mel = min_log_hz / f_sp; logstep = np.log(6.4) / 27.0
    return np.where(f >= min_log_hz, min_log_mel + np.log(np.maximum(f, 1e-10) / min_log_hz) / logstep, mels)


def mel_to_hz(m):
    m = np.asanyarray(m, dtype=float)
    f_sp = 200.0 / 3; freqs = f_sp * m
    min_log_hz = 1000.0; min_log_mel = min_log_hz / f_sp; logstep = np.log(6.4) / 27.0
    return np.where(m >= min_log_mel, min_log_hz * np.exp(logstep * (m - min_log_mel)), freqs)


def mel_filters(sr=SR, n_fft=N_FFT, n_mels=N_MELS):
    """Triangular filters with Slaney area normalisation (librosa.filters.mel defaults)."""
    fftfreqs = np.fft.rfftfreq(n_fft, 1.0 / sr)
    mel_f = mel_to_hz(np.linspace(hz_to_mel(0.0), hz_to_mel(sr / 2.0), n_mels + 2))
    fdiff = np.diff(mel_f); ramps = mel_f[:, None] - fftfreqs[None, :]
    w = np.zeros((n_mels, len(fftfreqs)))
    for i in range(n_mels):
        lower = -ramps[i] / fdiff[i]; upper = ramps[i + 2] / fdiff[i + 1]
        w[i] = np.maximum(0, np.minimum(lower, upper))
    w *= (2.0 / (mel_f[2:n_mels + 2] - mel_f[:n_mels]))[:, None]
    return w


def power_spec(y):
    """|STFT|² with centred frames and zero padding (librosa ≥ 0.10 default pad_mode='constant')."""
    y = np.pad(y, (N_FFT // 2, N_FFT // 2), mode="constant")
    n = 1 + (len(y) - N_FFT) // HOP
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(n)[:, None]
    return (np.abs(np.fft.rfft(y[idx] * WIN, axis=1)) ** 2).T


MEL = mel_filters(SR, N_FFT, N_MELS)
WIN = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(N_FFT) / N_FFT)       # periodic Hann, as in the pump runs
N_FRAMES = 1 + NS // HOP
FEATS = WORK / "logmel"; FEATS.mkdir(exist_ok=True)
LM = {}


def logmel_of(i_t):
    i, t = i_t
    return (10.0 * np.log10(MEL @ power_spec(f32(DST[t][i])) + EPS)).astype(np.float16)


t_feat = time.time()
for t in TYPES:
    f, n = FEATS / f"lm_{t}.npy", len(DMETA[t]["name"])
    if not f.exists():
        tmp = f.with_name(f.stem + ".part.npy")
        mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(n, N_MELS, N_FRAMES))
        for i, lm in enumerate(stream(logmel_of, [(i, t) for i in range(n)], depth=16, workers=max(2, (os.cpu_count() or 2)))):
            mm[i] = lm
        mm.flush(); del mm
        os.replace(tmp, f)
    LM[t] = np.load(f, mmap_mode="r")
    assert LM[t].shape == (n, N_MELS, N_FRAMES)
    print(f"{t}: {n} clips → log-mel {LM[t].shape} · {(time.time() - t_feat) / 60:.1f} min so far", flush=True)
FEAT_S = round(time.time() - t_feat, 1)


In [ ]:
# 5 · The four families (code/p04_mimii_models.py, unchanged classes) and the units they are fitted on. A unit is one machine type (each has one section):
# each family is fitted on that unit's normal training clips only, and scores that unit's test clips. No label is read here
def frames5(lm):
    """(64, T) log-mel → (T-4, 320): five consecutive frames side by side, as the baseline."""
    T = lm.shape[1]
    return np.concatenate([lm[:, t:T - 4 + t].T for t in range(5)], axis=1)


def clip_stats(logmel):
    x = logmel.astype(np.float32)
    return np.c_[x.mean(2), x.std(2)]


class AEfam:
    """M1, set up like the Keras baseline: no L2 penalty (Keras adds none), and the last 10 % of the
    training rows left out (Keras validation_split=0.1 holds them out before shuffling).
    Chosen on 2026-11-06 after the inspection (results/mimii/DECISION_after_inspection.md). It is also
    about five times faster: scikit-learn's default L2 slowly pushes dead-unit weights into float32
    denormals. The first run (alpha 0.0001, all rows) is kept in results/mimii/first_run/."""
    ALPHA, KEEP = 0.0, 0.9

    def fit(self, LM):
        X = np.concatenate([frames5(m.astype(np.float32)) for m in LM])
        X = X[: int(len(X) * self.KEEP)]                  # Keras: split_at = int(len(x) * (1 - validation_split))
        self.m = MLPRegressor(hidden_layer_sizes=(64, 64, 8, 64, 64), activation="relu", solver="adam", alpha=self.ALPHA,
                              batch_size=512, max_iter=50, n_iter_no_change=50, tol=0, shuffle=True,
                              random_state=0).fit(X, X)
        return self

    def score(self, LM):
        out = []
        for m in LM:
            X = frames5(m.astype(np.float32))
            out.append(float(np.mean((self.m.predict(X) - X) ** 2)))
        return np.array(out)


class Maha:
    """Standardise → PCA (keep 95 % or all) → T² + SPE, each over its 99th training percentile."""
    def __init__(self, var=0.95):
        self.var = var

    def fit(self, X):
        self.sc = StandardScaler().fit(X); Z = self.sc.transform(X)
        p = PCA().fit(Z)
        k = len(p.explained_variance_) if self.var is None else int(np.searchsorted(np.cumsum(p.explained_variance_ratio_), self.var) + 1)
        self.P, self.lam = p.components_[:k].T, p.explained_variance_[:k]
        t2, spe = self._parts(Z)
        self.qt, self.qs = np.quantile(t2, 0.99), max(np.quantile(spe, 0.99), 1e-12)
        return self

    def _parts(self, Z):
        T = Z @ self.P
        return (T ** 2 / self.lam).sum(1), ((Z - T @ self.P.T) ** 2).sum(1)

    def score(self, X):
        t2, spe = self._parts(self.sc.transform(X))
        return t2 / self.qt + spe / self.qs


class IFfam:
    def fit(self, X):
        self.sc = StandardScaler().fit(X)
        self.m = IsolationForest(n_estimators=200, random_state=0).fit(self.sc.transform(X))
        return self

    def score(self, X):
        return -self.m.score_samples(self.sc.transform(X))


def units(set_):
    return [(t, q) for t in TYPES for q in (DEV_IDS if set_ == "dev" else EVAL_IDS)[t]]


ALL_UNITS = [(s_, t, q) for s_ in ("dev", "eval") for t, q in units(s_)]


def unit_file(s_, t, q):
    return OUT / "scores" / f"{s_}_{t}_{q}.npz"


def load_unit(s_, t, q):
    f = unit_file(s_, t, q)
    return dict(np.load(f, allow_pickle=False)) if f.exists() else {}


def save_unit(s_, t, q, d):
    f = unit_file(s_, t, q); tmp = f.with_name(f.stem + ".part.npz")
    np.savez(tmp, **d); os.replace(tmp, f)


def fit_fast(s_, t, q):
    # families 1, 2 and 3: seconds per unit
    tr, te = dc_rows(t, s_, "train", q), dc_rows(t, s_, "test", q)
    A, B = np.asarray(LM[t][tr], dtype=np.float32), np.asarray(LM[t][te], dtype=np.float32)
    la, lb = A.mean((1, 2)), B.mean((1, 2))                           # one number per clip: its overall loudness, as on the pump page
    SA, SB = clip_stats(A), clip_stats(B)                             # 128 numbers per clip: the average and the wobble of each band
    return {"name": np.array(DMETA[t]["name"])[te], "F1": np.abs(lb - la.mean()) / max(float(la.std()), 1e-9), "F1_raw": lb,
            "F2": Maha(0.95).fit(SA).score(SB), "F3": IFfam().fit(SA).score(SB), "n_train": np.array(len(tr))}


def fit_ae(s_, t, q):
    tr, te = dc_rows(t, s_, "train", q), dc_rows(t, s_, "test", q)
    return AEfam().fit(np.asarray(LM[t][tr])).score(np.asarray(LM[t][te]))


t_fast = time.time()
for s_, t, q in ALL_UNITS:
    d = load_unit(s_, t, q)
    if "F3" not in d:
        save_unit(s_, t, q, {**d, **fit_fast(s_, t, q)})
FAST_S = round(time.time() - t_fast, 1)
print(f"families 1, 2 and 3: {len(ALL_UNITS)} units fitted and scored · {FAST_S / 60:.1f} min")


In [ ]:
# 6 · The official evaluator's numbers: per section AUC (source), AUC (target) and pAUC; the official score is the harmonic mean of every
# section's three values. hm, section_metrics and summary are notebook 30's functions, which the official evaluator reproduced there to 0.0
VALUES = ("AUC_source", "AUC_target", "pAUC")


def hm(v):
    # the harmonic mean, as the official evaluator takes it
    v = np.maximum(np.asarray(v, dtype=np.float64), sys.float_info.epsilon)
    return float(len(v) / np.sum(1.0 / v))


def section_metrics(y, s, dom):
    # the official evaluator's values of one section. AUC of a domain: that domain's normal clips against all the section's anomalous
    # clips, whatever their domain. pAUC: all the section's clips, max_fpr = 0.1
    y, s, dom = np.asarray(y), np.asarray(s, dtype=np.float64), np.asarray(dom)
    out = {"n_test": int(len(y)), "n_anomalies": int(y.sum())}
    for d in ("source", "target"):
        k = (dom == d) | (y != 0)
        out[f"AUC_{d}"] = float(roc_auc_score(y[k], s[k])); out[f"n_normal_{d}"] = int(((dom == d) & (y == 0)).sum())
    out["pAUC"] = float(roc_auc_score(y, s, max_fpr=0.1)); out["AUC_all"] = float(roc_auc_score(y, s))
    return out


def summary(per):
    # per: type → section → its values. The type's score is the harmonic mean of its sections' three values; the official score Ω is the
    # harmonic mean of every section's three values; the source, target and pAUC lines are harmonic means over the sections
    types = {t: {"score": hm([d[q][v] for q in d for v in VALUES]), **{v: hm([d[q][v] for q in d]) for v in VALUES}, "per_section": d} for t, d in per.items()}
    return {"omega": hm([d[q][v] for d in per.values() for q in d for v in VALUES]), **{v: hm([d[q][v] for d in per.values() for q in d]) for v in VALUES},
            "values": int(sum(len(d) * len(VALUES) for d in per.values())), "per_type": types}


def table(set_, labels):
    # labels: (type, section) → (array of 0/1, array of "source"/"target") in the order of that unit's saved names
    out = {}
    for f in SCORED:
        per = {}
        for t, q in units(set_):
            d = load_unit(set_, t, q)
            if f in d:
                per.setdefault(t, {})[q] = section_metrics(labels[(t, q)][0], d[f], labels[(t, q)][1])
        if sum(len(v) for v in per.values()) == len(units(set_)):
            out[f] = summary(per); out[f]["score"] = out[f]["omega"]
    return out


def show(res, title):
    ty = sorted({t for t, _ in units("dev")} | {t for t, _ in units("eval")}, key=TYPES.index)
    print(title); print(f"  {'':<62}{'official':>10}{'source':>9}{'target':>9}{'pAUC':>9}   " + "".join(f"{t[:11]:>12}" for t in ty if any(t in r["per_type"] for r in res.values())))
    for f in [f_ for f_ in SCORED if f_ in res]:
        r = res[f]
        print(f"  {FAM_NAME[f]:<62}{r['omega']:>10.4f}{r['AUC_source']:>9.4f}{r['AUC_target']:>9.4f}{r['pAUC']:>9.4f}   " + "".join(f"{v['score']:>12.4f}" for v in r["per_type"].values()))


def dev_labels():
    return {(t, q): (np.array(DMETA[t]["label"])[dc_rows(t, "dev", "test", q)], np.array(DMETA[t]["domain"])[dc_rows(t, "dev", "test", q)]) for t, q in units("dev")}


_m = section_metrics([0, 0, 0, 0, 1, 1], [0.1, 0.4, 0.3, 0.9, 0.5, 0.8], ["source", "source", "target", "target", "source", "target"])
assert (_m["AUC_source"], _m["AUC_target"], _m["AUC_all"]) == (1.0, 0.5, 0.75) and abs(hm([0.5, 1.0]) - 2 / 3) < 1e-12, "stop: the metrics are not the evaluator's"
DEV = table("dev", dev_labels())
show(DEV, "DCASE 2025 · development set · families 1 to 3 (family 4 follows in the next cell)")


In [ ]:
# 7 · Family 4, the autoencoder: one network per unit (the slow part). Each finished unit is saved in Drive › P04_SimpleFamilies_2025 › scores, so a
# session that stops is continued by Runtime › Run all
t_ae, done_now = time.time(), 0
todo = [u for u in ALL_UNITS if "F4" not in load_unit(*u)]
print(f"autoencoders to train: {len(todo)} of {len(ALL_UNITS)} units")
for k, (s_, t, q) in enumerate(todo):
    t1 = time.time(); d = load_unit(s_, t, q); d["F4"] = fit_ae(s_, t, q); d["F4_seconds"] = np.array(round(time.time() - t1, 1))
    save_unit(s_, t, q, d); done_now += 1
    left = (time.time() - t_ae) / done_now * (len(todo) - done_now)
    print(f"  {k + 1}/{len(todo)} · {s_} {t} {q}: {time.time() - t1:.0f} s · about {left / 60:.0f} min left", flush=True)
AE_S = round(sum(float(load_unit(*u)["F4_seconds"]) for u in ALL_UNITS), 1)
print(f"family 4: all {len(ALL_UNITS)} units done · training time in total {AE_S / 60:.0f} min")


In [ ]:
# 8 · The development set with all four families, then the evaluation set, scored once. Every score file is already written (cells 5
# and 7) and is locked here before a single evaluation label is read
assert all(all(f in load_unit(*u) for f in SCORED) for u in ALL_UNITS), "stop: a unit has no scores yet; run the cells above"
DEV = table("dev", dev_labels())
show(DEV, "DCASE 2025 · development set · the four simple families")
LOCK = OUT / "scores_written.json"
_sha = {f"{s_}_{t}_{q}": hashlib.sha256(unit_file(s_, t, q).read_bytes()).hexdigest() for s_, t, q in ALL_UNITS}
if LOCK.exists():
    assert json.loads(LOCK.read_text())["sha256"] == _sha, "stop: the score files changed after they were locked"
else:
    LOCK.write_text(json.dumps({"written": time.strftime("%Y-%m-%d %H:%M"), "sha256": _sha}, indent=1))


def fetch_evaluator():
    # the official evaluator's repository at the pinned commit: the script, its two tool files and the ground-truth files of the
    # evaluation set, checked against the recorded hashes before anything is used
    d = WORK / "evaluator2025"
    if not (d / "checked").exists():
        z = WORK / EV_ARCHIVE[1]
        download(EV_ARCHIVE[0], z)
        with zipfile.ZipFile(z) as zf:
            names = {n.split("/", 1)[1]: n for n in zf.namelist() if "/" in n and not n.endswith("/")}
            want = sorted(n for n in names if n.startswith(tuple(p + "/" for p in GT_PARTS)) and n.endswith(".csv"))
            code = [EV_SCRIPT] + EV_TOOLS
            if any(n not in names for n in code) or not want:
                raise SystemExit("stop: the evaluator's repository does not hold the script, its tools and the ground truth")
            blobs = {n: zf.read(names[n]) for n in code + want}
        h, hc = hashlib.sha256(), hashlib.sha256()
        for n in want:                                                # in path order
            h.update(n.encode() + b"\0" + blobs[n])
        for n in code:
            hc.update(n.encode() + b"\0" + blobs[n])
        if hc.hexdigest() != EV_SHA or h.hexdigest() != GT_SHA:
            raise SystemExit("stop: the evaluator's repository is not the recorded one (the script or a ground-truth file differs)")
        shutil.rmtree(d, ignore_errors=True)
        for n, b in blobs.items():
            (d / n).parent.mkdir(parents=True, exist_ok=True); (d / n).write_bytes(b)
        (d / "checked").write_text(f"{len(want)} ground-truth files")
    return d


def read_ground_truth():
    # the labels and the domains of the evaluation test clips: (type, section) → file name → (label, domain). The repository names a
    # clip as the evaluation zip does (section_00_NNNN.wav)
    d = fetch_evaluator(); gt = {}
    for t in TYPES_EVAL:
        for q in EVAL_IDS[t]:
            with open(d / GT_PARTS[0] / f"ground_truth_{t}_{q}_test.csv", newline="") as fh:
                lab = [r for r in csv.reader(fh) if r]
            with open(d / GT_PARTS[1] / f"ground_truth_{t}_{q}_test.csv", newline="") as fh:
                dom = [r for r in csv.reader(fh) if r]
            assert [r[0] for r in lab] == [r[0] for r in dom] and len(set(r[0] for r in lab)) == len(lab), (t, q)
            gt[(t, q)] = {a[0]: (int(float(a[1])), "target" if int(float(b[1])) == 1 else "source") for a, b in zip(lab, dom)}
    return d, gt


def eval_labels():
    _, gt = read_ground_truth(); out = {}
    for t, q in units("eval"):
        names = np.array(DMETA[t]["name"])[dc_rows(t, "eval", "test", q)].tolist()
        assert sorted(names) == sorted(gt[(t, q)]), f"{t} {q}: the clips are not the ground truth's"
        out[(t, q)] = (np.array([gt[(t, q)][n][0] for n in names]), np.array([gt[(t, q)][n][1] for n in names]))
    return out


EVAL = table("eval", eval_labels())                                  # the evaluation labels: read only here
show(EVAL, "DCASE 2025 · evaluation set · the four simple families")
print("\nbeside them, from the records (" + RECORDED_NOTE + "):")
for k, v in RECORDED.items():
    print(f"  {k:<62}{v:>10.4f}")


In [ ]:
# 9 · Save, and one zip to download
BEST = max(FAMS, key=lambda f: EVAL[f]["score"]); F5 = RECORDED["Family 5 · the kept system (expert + trainee)"]
RESULT = {"rules": RULES, "notebook": 33, "year": 2025, "date": time.strftime("%Y-%m-%d %H:%M"), "test_mode": TEST, "score": "the official score (harmonic mean)",
          "families": FAM_NAME, "settings": {"n_fft": N_FFT, "hop": HOP, "n_mels": N_MELS, "frames": int(N_FRAMES), "clip_samples": int(NS),
                                             "F2": "StandardScaler → PCA 95 % → T² + SPE", "F3": "IsolationForest, 200 trees, seed 0",
                                             "F4": "MLP 320-64-64-8-64-64-320, 50 epochs, batch 512, seed 0, first 90 % of the rows"},
          "units": {"dev": len(units("dev")), "eval": len(units("eval"))}, "development": DEV, "evaluation": EVAL, "recorded_beside": RECORDED,
          "recorded_note": RECORDED_NOTE,
          "answer": {"best_simple_family": BEST, "its_score": round(EVAL[BEST]["score"], 4), "family_5_kept_system": F5,
                     "family_5_less_best_simple": round(F5 - round(EVAL[BEST]["score"], 4), 4),
                     "a_simple_family_reaches_family_5": bool(round(EVAL[BEST]["score"], 4) >= F5)},
          "clips": {t: len(DMETA[t]["name"]) for t in TYPES}, "zips": {t: DMETA[t]["zips"] for t in TYPES},
          "seconds": {"log_mel": FEAT_S, "families_1_2_3": FAST_S, "family_4": AE_S, "session": round(time.time() - T_START, 1)}}
(OUT / "simple_families_2025.json").write_text(json.dumps(RESULT, indent=1))
print(json.dumps(RESULT["answer"], indent=1))
Z = OUT / "p04_n33_results.zip"
with zipfile.ZipFile(Z, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / "simple_families_2025.json", "simple_families_2025.json"); zf.write(LOCK, LOCK.name)
    for u in ALL_UNITS:
        zf.write(unit_file(*u), f"scores/{unit_file(*u).name}")
print("saved:", Z, f"({Z.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(Z))


## Send back
Send the zip that downloaded (`p04_n33_results.zip`). It is also in Drive › `P04_SimpleFamilies_2025`.